# Vizualise cell assemblies reactivation

Load packages

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, Cursor
from scipy import fftpack
import pandas as pd
from pathlib import Path
import os
import copy
import json
from IPython.display import display
from ipyfilechooser import FileChooser
from ephyviewer import mkQApp, MainViewer, TraceViewer
from ephyviewer import AnalogSignalSourceWithScatter
import ephyviewer
from scipy.stats import zscore
from scipy.interpolate import interp1d
from itertools import groupby
import sys 
import pickle
import seaborn as sns
from scipy.signal import find_peaks
from scipy.signal import chirp, find_peaks, peak_widths
from scipy import stats
import matplotlib.colors as mcolors
import warnings
from scipy import interpolate
import re
from scipy.stats import f_oay
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu
from scipy.stats import ks_2samp
import diptest
import random
from collections import Counter, defaultdict
import seaborn as sns
import pandas as pd
from matplotlib.colors import ListedColormap
import networkx as nx

from scipy.stats import f_oay
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import ast
warnings.filterwarnings("ignore")

#%matplotlib widget

## Cell assemblies activity across vigilance states

Plot Cell assembly activity relative to Vig States

In [ ]:
Analysisfolder= 'Reactivation_2025-10-27_10_10_37_spikes_200ms' 

In [ ]:
dfilepath=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{Analysisfolder}/CellAssembly_Global.pkl"
with open(dfilepath, 'rb') as pickle_file:
    df_CellAss_origin = pickle.load(pickle_file)

drug= 'baseline'
NrSubtype='L2_3_mice' # L1NDNF_mice OR L2_3_mice

df_CellAss=df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]

df_CellAss = df_CellAss[df_CellAss['ExpeType']=='SleepAfter']

initial_nb= df_CellAss['Assembly_ID'].nunique()
df_CellAss = df_CellAss[df_CellAss['Assembly_size'] > 1]
print(initial_nb - df_CellAss['Assembly_ID'].nunique(),'/',initial_nb, 'cell assemblies contained no neurons')

# Convert string representations of lists into actual Python lists.
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Filter the DataFrame to only keep rows where the number of cells in the list matches the 'Assembly_size'.
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]

plt.figure(figsize=(6, 6))
table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
desired_order = ['AW','QW', 'NREM', 'REM']   
try:del table_CellAss['undefined']
except: pass
try:del table_CellAss['IS']
except: pass
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(2,2,1)
plt.plot(table_CellAss.columns, table_CellAss.values.T, alpha=0.5, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
             fmt='o', color='black', capsize=5)
plt.ylabel('Reactivation strength per cell assemblies')
plt.title(f'{NrSubtype}, n={len(table_CellAss)}, {drug}')
plt.tight_layout()

groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
df_melt = table_CellAss.melt(var_name='Substates', value_name='Avg_Activity')
df_melt = df_melt.dropna(subset=['Avg_Activity', 'Substates'])
df_melt['Avg_Activity'] = pd.to_numeric(df_melt['Avg_Activity'], errors='coerce')
tukey = pairwise_tukeyhsd(endog=df_melt['Avg_Activity'], groups=df_melt['Substates'], alpha=0.05)
print(tukey.summary())

table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='EventFreq', aggfunc='mean', fill_value=None)
try:del table_CellAss['undefined']
except: pass
try:del table_CellAss['IS']
except: pass
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(2,2,3)
plt.plot(table_CellAss.columns, table_CellAss.values.T, alpha=0.5, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
            fmt='o', color='black', capsize=5)
plt.ylabel('Event frequency per cell assemblies')

groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
df_melt = table_CellAss.melt(var_name='Substates', value_name='EventFreq')
df_melt = df_melt.dropna(subset=['EventFreq', 'Substates'])
df_melt['EventFreq'] = pd.to_numeric(df_melt['EventFreq'], errors='coerce')
tukey = pairwise_tukeyhsd(endog=df_melt['EventFreq'], groups=df_melt['Substates'], alpha=0.05)
print(tukey.summary())

plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_VigStActivity_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Load NeuronIdentity file & assign Neuron ID to each neuron

In [ ]:
AnalysisfolderVig= 'VigSt_2025-10-25_16_35_21'
AnalysisfolderNeuronID= 'NeuronIdentity_2025-10-10_11_13_40'

In [ ]:
with open(f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{AnalysisfolderNeuronID}/List_SignCells.pkl", 'rb') as pickle_file:
    df_MazeCellID = pickle.load(pickle_file)

AllCheeseboard_cells=np.array(df_MazeCellID['all'])
HD_cells=np.array(df_MazeCellID['HD'])
PC_cells=np.array(df_MazeCellID['PC'])
PCHD_cells=np.array(df_MazeCellID['PC&HD'])

Spatial_cells=np.concatenate([PCHD_cells,  HD_cells, PC_cells])

dfilepathV=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{AnalysisfolderVig}/VigStates_Global.pkl"
with open(dfilepathV, 'rb') as pickle_file:
    df_VigSt = pickle.load(pickle_file)

All_cells= np.array(df_VigSt['Unit_ID'].unique().tolist())

id_to_cluster = {
    p: (
        'HD' if p in HD_cells.tolist()
        else 'PC' if p in PC_cells.tolist()
        else 'PC&HD' if p in PCHD_cells.tolist()
        else 'notPCnotHD' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}
notPCnotHD_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'notPCnotHD'])
unclassified_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'unclassified'])


Spatial_id_to_cluster = {
    p: (
        'Spatial' if p in Spatial_cells.tolist()
        else 'Non_Spatial' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}

Non_Spatial_cells = np.array([name for name, classification in Spatial_id_to_cluster.items() if classification == 'Non_Spatial'])

Save Vigilance State with Neuron ID

In [ ]:
df_VigSt['Neuron_Identity'] = df_VigSt['Unit_ID'].map(id_to_cluster)
df_VigSt['Spatially_tuned'] = df_VigSt['Unit_ID'].map(Spatial_id_to_cluster)

filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_VigSt, pickle_file)
    
filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_VigSt.to_excel(writer, index=False)

Identify Cell assembly Types and test proportion

In [ ]:
df_CellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_CellAss_uniqAss = df_CellAss_uniqAss[df_CellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_CellAss_uniqAss['Assembly_size']]
groups = df_CellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=AllCheeseboard_cells.tolist()
cluster_labels = ['HD']*len(HD_cells) + ['PC']*len(PC_cells) + ['PC&HD']*len(PCHD_cells) + ['notPCnotHD']*len(notPCnotHD_cells) 

def classify_groups(groups, id_to_cluster):
    labels = []
    for group in groups:
        clusters = [id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, id_to_cluster)
observed_counts = Counter(observed_labels)
df_CellAss_uniqAss['Assembly_Neurons_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100

df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])
fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue',  'green', 'orange','black','purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1 = ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2 = ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'mix' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_CellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_SpCellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_SpCellAss_uniqAss = df_SpCellAss_uniqAss[df_SpCellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_SpCellAss_uniqAss['Assembly_size']]
groups = df_SpCellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=AllCheeseboard_cells.tolist()
cluster_labels = ['Spatial']*len(Spatial_cells) + ['Non_Spatial']*len(Non_Spatial_cells)

def classify_groups(groups, Spatial_id_to_cluster):
    labels = []
    for group in groups:
        clusters = [Spatial_id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, Spatial_id_to_cluster)
observed_counts = Counter(observed_labels)
df_SpCellAss_uniqAss['Assembly_Spatial_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial","mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "mix"])

fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue', 'orange', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1= ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2= ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'mix' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_SpCellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/SpatialCellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Add & save Assembly main identity

In [ ]:
df_CellAss_ID_ = pd.merge(df_CellAss,df_CellAss_uniqAss[['Assembly_ID', 'Assembly_Neurons_ID']], on='Assembly_ID', how='outer')
df_CellAss_ID = pd.merge(df_CellAss_ID_,df_SpCellAss_uniqAss[['Assembly_ID', 'Assembly_Spatial_ID']], on='Assembly_ID', how='outer')

filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_CellAss_ID, pickle_file)    
filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_CellAss_ID.to_excel(writer, index=False)

### HD vs PC vs HDPC vs notHDnotPC

Plot Reactivation of cell assemblies

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(13,3))
axs = axs.flatten()
substates= ['SleepAfter', 'AW', 'QW', 'NREM', 'REM']
for plot, substate in enumerate(substates): 
    ax= axs[plot]

    df_CellAss_ID_ = df_CellAss_ID[(df_CellAss_ID['Substate'] == substate)] if not substate =='SleepAfter' else df_CellAss_ID
    CellAss_ID_pivot= df_CellAss_ID_.pivot_table(index='Assembly_ID', columns=['Assembly_Neurons_ID'], values='Avg_ReactStr', aggfunc='mean', fill_value=None)

    g = pd.DataFrame({"mean": np.nanmean(CellAss_ID_pivot, axis=0),"sem": CellAss_ID_pivot.sem(skipna=True),"count" : CellAss_ID_pivot.count()})
    g = g.reindex(['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix'])   
    color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'black','mix': 'purple'} 
    colors = [color_map[c] for c in g.index]   # in order of shown categories
    for x,(m,e,c) in enumerate(zip(g['mean'], g['sem'], colors)):
        ax.errorbar(x, m, yerr=e, fmt='o', capsize=5, color=c)
    ax.set_ylabel('Reactivation Strength')
    ax.axhline(y=0, linestyle='--', color='grey')
    ax.set_xticks(range(len(g)), g.index, rotation=45, ha='right')
    ax.set_yticks((-.15,.15))
    ax.set_title(f"During {substate} \n (n = {g['count'].values.tolist()})")
    
    print(f'############################ {substate} ############################')    
    groups = [CellAss_ID_pivot[col].dropna().values for col in CellAss_ID_pivot.columns]
    f_stat, p_value = f_oneway(*groups)
    print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
    if p_value<0.05:
        df_melt = CellAss_ID_pivot.melt(var_name='Assembly_Neurons_ID', value_name='Avg_ReactStr')
        df_melt = df_melt.dropna(subset=['Avg_ReactStr', 'Assembly_Neurons_ID'])
        df_melt['Avg_ReactStr'] = pd.to_numeric(df_melt['Avg_ReactStr'], errors='coerce')
        tukey = pairwise_tukeyhsd(endog=df_melt['Avg_ReactStr'], groups=df_melt['Assembly_Neurons_ID'], alpha=0.05)
        print(tukey.summary())

plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyRS_perVigSt&NeuronID.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
CellAss_ID_pivot= df_CellAss_ID.pivot_table(index='Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
CellAss_ID_pivot_clean = pd.merge(CellAss_ID_pivot,df_CellAss_ID[['Assembly_Neurons_ID', 'Assembly_ID']], on='Assembly_ID', how='inner')
CellAss_ID_pivot_clean = CellAss_ID_pivot_clean.drop_duplicates()

color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 
             'notPCnotHD': 'black', 'mix': 'purple'}

df_melted = CellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Neurons_ID',
    value_vars=['SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Neurons_ID'] = pd.Categorical(df_melted['Assembly_Neurons_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])

fig, ax= plt.subplots(figsize=(15,2))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Neurons_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0], ['Cheeseboard'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySize_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Neurons_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Neurons_ID', values='count').fillna(0)
size = 0.5
vals_outer = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'green', 'orange', 'black', 'purple']

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard')
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyProportion_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

### Spatially-tuned vs non Spatially-tuned

Plot Reactivation of cell assemblies

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10,3))
axs = axs.flatten()
substates= ['SleepAfter', 'AW', 'QW', 'NREM', 'REM']
for plot, substate in enumerate(substates): 
    ax= axs[plot]

    df_CellAss_ID_ = df_CellAss_ID[(df_CellAss_ID['Substate'] == substate)] if not substate =='SleepAfter' else df_CellAss_ID
    CellAss_ID_pivot= df_CellAss_ID_.pivot_table(index='Assembly_ID', columns=['Assembly_Spatial_ID'], values='Avg_ReactStr', aggfunc='mean', fill_value=None)

    g = pd.DataFrame({"mean": np.nanmean(CellAss_ID_pivot, axis=0),"sem": CellAss_ID_pivot.sem(skipna=True),"count" : CellAss_ID_pivot.count()})
    g = g.reindex(['Spatial', 'Non_Spatial','mix'])   
    color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'} 
    colors = [color_map[c] for c in g.index]   # in order of shown categories
    for x,(m,e,c) in enumerate(zip(g['mean'], g['sem'], colors)):
        ax.errorbar(x, m, yerr=e, fmt='o', capsize=5, color=c)
    ax.set_ylabel('Reactivation Strength')
    ax.axhline(y=0, linestyle='--', color='grey')
    ax.set_xticks(range(len(g)), g.index, rotation=45, ha='right')
    ax.set_yticks((-.15,.15))
    ax.set_title(f"During {substate} \n (n = {g['count'].values.tolist()})")
    
    print(f'############################ {substate} ############################')    
    groups = [CellAss_ID_pivot[col].dropna().values for col in CellAss_ID_pivot.columns]
    f_stat, p_value = f_oneway(*groups)
    print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
    if p_value<0.05: 
        df_melt = CellAss_ID_pivot.melt(var_name='Assembly_Spatial_ID', value_name='Avg_ReactStr')
        df_melt = df_melt.dropna(subset=['Avg_ReactStr', 'Assembly_Spatial_ID'])
        df_melt['Avg_ReactStr'] = pd.to_numeric(df_melt['Avg_ReactStr'], errors='coerce')
        tukey = pairwise_tukeyhsd(endog=df_melt['Avg_ReactStr'], groups=df_melt['Assembly_Spatial_ID'], alpha=0.05)
        print(tukey.summary())

plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyRS_perVigSt&SpatialID.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
CellAss_ID_pivot = df_CellAss_ID.pivot_table(index='Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
CellAss_ID_pivot_clean = pd.merge(CellAss_ID_pivot,df_CellAss_ID[['Assembly_Spatial_ID', 'Assembly_ID']], on='Assembly_ID', how='inner')
CellAss_ID_pivot_clean = CellAss_ID_pivot_clean.drop_duplicates()

color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'} 

df_melted = CellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Spatial_ID',
    value_vars=['SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Spatial_ID'] = pd.Categorical(df_melted['Assembly_Spatial_ID'], ["Spatial", "Non_Spatial", "mix"])

fig, ax= plt.subplots(figsize=(15,2))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Spatial_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0], ['Cheeseboard'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySize_Spatial_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Spatial_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Spatial_ID', values='count').fillna(0)
size = .5
vals_outer = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'orange', 'purple',]

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard')
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyProportion_Spatial_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()